## Example 1: parse function prototypes and variable declarations from a C header

In [ ]:
# Example 1: parse function prototypes and variable declarations from a C header
import importlib.metadata
import os
import tempfile

from pyclibrary import CParser

print(f"pyclibrary {importlib.metadata.version('pyclibrary')}")

header = """
/* A small API: two functions, one taking a pointer, and two global variables. */
int add(int a, int b);
double scale(const double *values, unsigned long count, double factor);
extern int call_count;
static const int LIMIT = 10;
"""
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "api.h")
    with open(path, "w") as f:
        f.write(header)
    parser = CParser([path])

functions = parser.defs["functions"]
print("functions:", functions)
assert sorted(functions) == ["add", "scale"], sorted(functions)

# A function's type is (return type, ((name, type, default), ...)).
add_ret, add_args = functions["add"]
assert add_ret == ("int",), add_ret
assert [(name, typ) for name, typ, _ in add_args] == [("a", ("int",)), ("b", ("int",))], add_args
scale_ret, scale_args = functions["scale"]
assert scale_ret == ("double",) and [name for name, _, _ in scale_args] == ["values", "count", "factor"], functions["scale"]
assert scale_args[0][1] == ("double", "*") and scale_args[1][1] == ("unsigned long",), scale_args

variables = parser.defs["variables"]
print("variables:", variables)
assert variables["call_count"][0] is None and variables["call_count"][1] == ("int",), variables["call_count"]
# An initialised constant keeps its value, which is also published under defs["values"].
assert variables["LIMIT"][0] == 10 and parser.defs["values"]["LIMIT"] == 10, variables["LIMIT"]

## Example 2: macros, function-like macros and conditional compilation

In [ ]:
# Example 2: macros, function-like macros and conditional compilation
import os
import tempfile

from pyclibrary import CParser

header = """
#define VERSION_MAJOR 2
#define VERSION_MINOR 7
#define VERSION ((VERSION_MAJOR << 8) | VERSION_MINOR)
#define GREETING "hello"
#define SQUARE(x) ((x) * (x))

#ifdef VERSION_MAJOR
#define HAVE_MAJOR 1
#else
#define HAVE_MAJOR 0
#endif

#if VERSION_MINOR > 10
#define LARGE_MINOR 1
#endif
"""
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "config.h")
    with open(path, "w") as f:
        f.write(header)
    parser = CParser([path])

macros = parser.defs["macros"]
values = parser.defs["values"]
print("macros:", macros)
print("values:", values)

# Object-like macros are expanded, then evaluated into Python values.
assert macros["VERSION"] == "((2 << 8) | 7)", macros["VERSION"]
assert values["VERSION"] == (2 << 8) | 7 == 519, values["VERSION"]
assert values["GREETING"] == "hello", values["GREETING"]

# #ifdef / #if blocks are evaluated: only the taken branch defines anything.
assert values["HAVE_MAJOR"] == 1, values["HAVE_MAJOR"]
assert "LARGE_MINOR" not in macros, sorted(macros)

# Function-like macros are kept separately and are not evaluated on their own.
assert "SQUARE" in parser.defs["fnmacros"] and "SQUARE" not in macros, parser.defs["fnmacros"]

## Example 3: structs, unions and typedefs

In [ ]:
# Example 3: structs, unions and typedefs
import os
import tempfile

from pyclibrary import CParser

header = """
typedef unsigned int uint32;
typedef struct point { int x; int y; } point_t;
struct rect { point_t origin; uint32 width; uint32 height; };
union value { int i; float f; char bytes[4]; };
"""
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "shapes.h")
    with open(path, "w") as f:
        f.write(header)
    parser = CParser([path])

structs = parser.defs["structs"]
unions = parser.defs["unions"]
types = parser.defs["types"]
print("structs:", structs)
print("unions:", unions)
print("types:", types)

assert sorted(structs) == ["point", "rect"], sorted(structs)
# Each member is (name, type, bit width or None).
assert [(m[0], m[1]) for m in structs["point"]["members"]] == [("x", ("int",)), ("y", ("int",))], structs["point"]
assert [m[0] for m in structs["rect"]["members"]] == ["origin", "width", "height"], structs["rect"]
assert structs["rect"]["members"][0][1] == ("point_t",), structs["rect"]["members"][0]

assert [m[0] for m in unions["value"]["members"]] == ["i", "f", "bytes"], unions["value"]
assert unions["value"]["members"][2][1] == ("char", [4]), unions["value"]["members"][2]

# Typedefs resolve to the underlying type.
assert types["uint32"] == ("unsigned int",), types["uint32"]
assert types["point_t"] == ("struct point",), types["point_t"]
assert parser.eval_type(["point_t"]) == ("struct point",), parser.eval_type(["point_t"])

## Example 4: named and anonymous enums

In [ ]:
# Example 4: named and anonymous enums
import os
import tempfile

from pyclibrary import CParser

header = """
enum color { RED, GREEN = 5, BLUE };
typedef enum { LOW = -1, MEDIUM, HIGH = MEDIUM + 10 } level_t;
"""
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "enums.h")
    with open(path, "w") as f:
        f.write(header)
    parser = CParser([path])

enums = parser.defs["enums"]
print("enums:", enums)

# Values follow C rules: start at 0, and each implicit value is the previous one plus 1.
assert enums["color"] == {"RED": 0, "GREEN": 5, "BLUE": 6}, enums["color"]

# An anonymous enum gets a generated name, and the typedef points at it.
anonymous = [name for name in enums if name != "color"]
assert len(anonymous) == 1, sorted(enums)
assert enums[anonymous[0]] == {"LOW": -1, "MEDIUM": 0, "HIGH": 10}, enums[anonymous[0]]
assert parser.defs["types"]["level_t"] == ("enum " + anonymous[0],), parser.defs["types"]["level_t"]

# Enumerators are also published as plain values.
values = parser.defs["values"]
assert (values["BLUE"], values["HIGH"]) == (6, 10), values

## Example 5: call C math functions through CLibrary

In [ ]:
# Example 5: call C math functions through CLibrary
import math
import os
import tempfile

from pyclibrary import CLibrary, CParser

# Declarations for three functions of the C math library. CLibrary uses them to convert
# arguments and return values; frexp's int* argument is an output that CLibrary allocates.
header = """
double cos(double x);
double ldexp(double x, int exp);
double frexp(double x, int *exp);
#define ONE_HALF 0.5
"""
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "m.h")
    with open(path, "w") as f:
        f.write(header)
    parser = CParser([path])

# "m" is looked up like the linker's -lm.
libm = CLibrary("m", parser)
print(libm)

result = libm.cos(0.0)
print("cos(0.0) =", result.rval)
assert result.rval == 1.0, result.rval
assert math.isclose(libm.cos(math.pi).rval, -1.0), libm.cos(math.pi).rval

# Calling a CallResult returns the return value; arguments are converted from Python numbers.
assert libm.ldexp(0.75, 4)() == 12.0

# frexp(8.0) = 0.5 * 2**4: the mantissa is returned and the exponent comes back through the pointer.
mantissa_exponent = libm.frexp(8.0)
print("frexp(8.0) =", mantissa_exponent.rval, mantissa_exponent["exp"])
assert mantissa_exponent.rval == 0.5 and mantissa_exponent["exp"] == 4 and mantissa_exponent[1] == 4

# Macro values from the header are available on the library object too.
assert libm.ONE_HALF == 0.5, libm.ONE_HALF